<div style="text-align:center;"><h1>Assignment 03: Advanced NLP Tasks with Hugging Face Pipelines</h1></div>

The goal of this assignment is to deepen your understanding of Hugging Face pipelines,
explore custom models, and demonstrate how to use these pipelines for various
advanced NLP tasks including text classification, question answering, and text
generation.

**Group** :

- **Gleb Ignatov**
- **Liam Knapp**
- **Gautam Singh**
- **Minh Le Nguyen**

**Note**

- **Google Drive Link**:https://drive.google.com/drive/folders/1Es8-g-wy0HPSNrJfHdq5H-pscaoapdqW?usp=sharing

**Documents**

- <a>https://huggingface.co/docs/transformers/en/add_new_pipeline</a>
- <a>https://huggingface.co/docs/transformers/training</a>
- <a>https://huggingface.co/docs/transformers/en/quicktour</a>

**All Models**

- <a>https://huggingface.co/transformers/v3.3.1/pretrained_models.html</a>
- <a>https://huggingface.co/models</a>

## A. Set Up Environment

### I. Check if pytorch is installed

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

CUDA available: True
CUDA version: 12.4


### II. Install pytorch, keras (required), Hugging Face transformer for torch and datasets

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install transformers[torch]
!pip install tf-keras
!pip install datasets
!pip install -U datasets huggingface_hub fsspec

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidi

---

## B. Implementation Tasks

### Task I. Create a sentiment analysis pipeline and classify a few sample texts (Based Model - 2 Label Positive, Negative)

Write a Python script to demonstrate the basic usage of a pipeline for text
classification. Use the pipeline function to create a sentiment analysis pipeline and classify a few sample texts.

#### 1. Train a custom (pre-trained) text classification model using the Hugging Face library. Use a dataset such as IMDb for sentiment analysis

##### i. Import Necessary Library

In [3]:
from datasets import (load_dataset , Dataset)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import numpy as np

##### ii. Retrieve IMDB Dataset

In [4]:
raw_imdb_datasets = load_dataset("imdb")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

##### iii. Extract the train and test portion from the raw dataset

In [5]:
raw_imdb_datasets_train = raw_imdb_datasets['train']
raw_imdb_datasets_test = raw_imdb_datasets['test']

In [6]:
raw_imdb_datasets_train

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [7]:
raw_imdb_datasets_test

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

##### iv. Prepare the tokenizer and model

<a>https://huggingface.co/google-bert/bert-base-uncased</a>

In [8]:
bert_base_model = "bert-base-uncased"
bert_base_tokenizer = AutoTokenizer.from_pretrained(bert_base_model)
bert_base_model_for_sequence_classification = AutoModelForSequenceClassification.from_pretrained(bert_base_model, num_labels=2)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### 2. Fine-tune the model and save it.

##### v. Tokenize the train and test datasets

In [9]:
def tokenize_batch(batch):
    return bert_base_tokenizer(batch['text'], padding='max_length', truncation=True, max_length=512)

imdb_tokenized_train = raw_imdb_datasets_train.map(tokenize_batch, batched=True)
imdb_tokenized_test  = raw_imdb_datasets_test.map(tokenize_batch, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

##### vi. Set up the training arguments

**Note**

- Change the value of per_device_train_batch_size and per_device_eval_batch_size to 32 or smaller if not have enough memory to train it

In [10]:
bert_base_training_args = TrainingArguments(
    output_dir = "./bert_base_sentiment_model",
    num_train_epochs = 1,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 64,
    save_strategy = "epoch",
    logging_dir = "./bert_base_logs",
    logging_steps = 200,
    learning_rate = 2e-5,
    fp16=False,
    dataloader_num_workers=8,
)

##### vii. Set up the function to compute metrics

In [11]:
def bert_base_compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}

##### viii. Create Trainer and fine tuning the Bert-based model

In [12]:
bert_base_trainer = Trainer(
    model = bert_base_model_for_sequence_classification,
    args  = bert_base_training_args,
    train_dataset = imdb_tokenized_train,
    eval_dataset = imdb_tokenized_test,
    tokenizer = bert_base_tokenizer,
    compute_metrics = bert_base_compute_metrics
)

bert_base_trainer.train()

/tmp/ipython-input-12-4237825419.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  bert_base_trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: minhlenguyen7 (minhlenguyennolanm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
200,0.302700


TrainOutput(global_step=391, training_loss=0.25340956861100844, metrics={'train_runtime': 471.4382, 'train_samples_per_second': 53.029, 'train_steps_per_second': 0.829, 'total_flos': 6577776384000000.0, 'train_loss': 0.25340956861100844, 'epoch': 1.0})

##### ix. Save the fine tuned model and the tokenizer

In [13]:
bert_base_trainer.save_model("./bert_base_sentiment_model")
bert_base_tokenizer.save_pretrained("./bert_base_sentiment_model")

('./bert_base_sentiment_model/tokenizer_config.json',
 './bert_base_sentiment_model/special_tokens_map.json',
 './bert_base_sentiment_model/vocab.txt',
 './bert_base_sentiment_model/added_tokens.json',
 './bert_base_sentiment_model/tokenizer.json')

#### 3. Use the trained model with the Hugging Face pipeline function to classify texts.

##### X. Reload the fine tuned model into a pipeline

In [14]:
bert_base_sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model      = "./bert_base_sentiment_model",
    tokenizer  = "./bert_base_sentiment_model",
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=128
)

Device set to use cuda


##### XI. Testing the Bert based fine tuned model

In [15]:
base_bert_testing_samples = [
    "I absolutely loved this movie! The performances were stellar and the story was gripping.",
    "Terrible film. It was too long, the plot made no sense, and I fell asleep halfway through.",
    "It was an okay experience – some parts were enjoyable, others not so much.",
    "I do not like this ice-scream that much",
    "Don’t forget to update your data",
    "No refund",
    "I have no bad day at all",
    "I have a great day. Let get some Tim Horton",
    "I have a bad assignment grade"
]
base_bert_testing_dataset = Dataset.from_dict({"text": base_bert_testing_samples})

---

In [16]:
def predict_batch(batch):
    outputs = bert_base_sentiment_analyzer(batch["text"])
    batch["label"] = [out["label"] for out in outputs]
    batch["score"] = [out["score"] for out in outputs]
    return batch

#### 4. Print classification results.

In [17]:
base_bert_testing_dataset = base_bert_testing_dataset.map(predict_batch, batched=True, batch_size=128)

label_map = {"LABEL_0": "Negative", "LABEL_1": "Positive"}
for row in base_bert_testing_dataset:
    human_label = label_map.get(row["label"], row["label"])
    print(f'Text: "{row["text"]}"')
    print(f'Label: {human_label} ({row["label"]})')
    print(f'Score: {row["score"]:.3f}\n')

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Text: "I absolutely loved this movie! The performances were stellar and the story was gripping."
Label: Positive (LABEL_1)
Score: 0.990

Text: "Terrible film. It was too long, the plot made no sense, and I fell asleep halfway through."
Label: Negative (LABEL_0)
Score: 0.969

Text: "It was an okay experience – some parts were enjoyable, others not so much."
Label: Positive (LABEL_1)
Score: 0.905

Text: "I do not like this ice-scream that much"
Label: Negative (LABEL_0)
Score: 0.736

Text: "Don’t forget to update your data"
Label: Positive (LABEL_1)
Score: 0.895

Text: "No refund"
Label: Positive (LABEL_1)
Score: 0.794

Text: "I have no bad day at all"
Label: Positive (LABEL_1)
Score: 0.775

Text: "I have a great day. Let get some Tim Horton"
Label: Positive (LABEL_1)
Score: 0.969

Text: "I have a bad assignment grade"
Label: Negative (LABEL_0)
Score: 0.817



### Task II. create a custom question-answering model

Write a short Python script to demonstrate the basic usage of a pipeline for
question-answering NLP tasks. Use the pipeline function to create a custom question-answering model.

#### 1. Train a custom (pre-trained) question-answering model using the Hugging Face library. Use Stanford Question Answering Dataset SQuAD

##### i. Import Necessary Library

In [18]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    pipeline
)

##### ii. Retrieve SQuAD datasets

In [19]:
raw_squad_datasets = load_dataset("squad")
raw_squad_datasets

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

##### iii. Extract train and validate data portions from the raw dataset

In [20]:
raw_squad_datasets_train = raw_squad_datasets['train']
raw_squad_datasets_test = raw_squad_datasets['validation']

In [21]:
raw_squad_datasets_train

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

In [22]:
raw_squad_datasets_test

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10570
})

##### iv. Prepare the tokenizer and model for SQuAD dataset

<a>https://huggingface.co/distilbert/distilbert-base-uncased</a>

In [23]:
model_squad_name = "distilbert-base-uncased"
squad_tokenizer = AutoTokenizer.from_pretrained(model_squad_name)
squad_model = AutoModelForQuestionAnswering.from_pretrained(model_squad_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
squad_model

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

#### 2. Fine-tune the model and save it.

##### v. Tokenize the train and test datasets

In [25]:
def prepare_train_features(examples):
    # Tokenize question and context
    tokenized = squad_tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=350,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    # Loop over each feature
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        sequence_ids = tokenized.sequence_ids(i)
        cls_index = input_ids.index(squad_tokenizer.cls_token_id)

        # Map feature to original sample
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # If no answer, set to CLS index
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Find the start of the context tokens
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # If answer is out of the limit, return CLS
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                # Move token_start_index to the exact start token
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                # Move token_end_index to the exact end token
                while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1

                # Final positions
                start_positions.append(token_start_index - 1)
                end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"]   = end_positions
    return tokenized

In [26]:
squad_tokenized_train = raw_squad_datasets_train.map(prepare_train_features, batched=True, remove_columns=raw_squad_datasets_train.column_names)
squad_tokenized_test = raw_squad_datasets_test.map(prepare_train_features, batched=True, remove_columns=raw_squad_datasets_test.column_names)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [27]:
squad_tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 87714
})

In [28]:
squad_tokenized_test

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 10626
})

##### vi. Set up the training arguments

In [29]:
training_args = TrainingArguments(
    output_dir="./qa_distilbert_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    save_strategy="epoch",
    logging_steps=1000,
    dataloader_num_workers=8,
)

##### vii. Set up the function to compute metrics

##### viii. Create Trainer and fine tuning the Distilbert Base Uncased Distilled Squad model

In [30]:
squad_trainer = Trainer(
    model=squad_model,
    args=training_args,
    train_dataset=squad_tokenized_train,
    eval_dataset=squad_tokenized_test,
    tokenizer=squad_tokenizer,
)
squad_trainer.train()

/tmp/ipython-input-30-2201926080.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  squad_trainer = Trainer(


Step,Training Loss
1000,1.655100


TrainOutput(global_step=1371, training_loss=1.5419942131014657, metrics={'train_runtime': 776.5949, 'train_samples_per_second': 112.947, 'train_steps_per_second': 1.765, 'total_flos': 1.1460106016206848e+16, 'train_loss': 1.5419942131014657, 'epoch': 1.0})

##### ix. Save the fine tuned model and the tokenizer

In [31]:
squad_trainer.save_model("./qa_distilbert_model")
squad_tokenizer.save_pretrained("./qa_distilbert_model")

('./qa_distilbert_model/tokenizer_config.json',
 './qa_distilbert_model/special_tokens_map.json',
 './qa_distilbert_model/vocab.txt',
 './qa_distilbert_model/added_tokens.json',
 './qa_distilbert_model/tokenizer.json')

#### 3. Use the trained model with the pipeline function to answer questions based on a given context.

##### X. Reload the fine tuned model into a pipeline

In [32]:
squad_qa_pipeline = pipeline(
    "question-answering",
    model="./qa_distilbert_model",
    tokenizer="./qa_distilbert_model",
    device= "cuda" if torch.cuda.is_available() else "cpu"
)

Device set to use cuda


##### XI. Quick testing the Distilbert Base Uncased Distilled Squad fine tuned model

In [33]:
context = (
  "The Transformers library provides thousands of pretrained models"
  " to perform tasks on texts such as classification, information extraction,"
  " question answering, summarization, translation, and more."
)
question = "What tasks can the Transformers library perform?"

result = squad_qa_pipeline({"context": context, "question": question})
print(f"Answer: {result['answer']}")
print(f"Score: {result['score']:.4f}")

Answer: classification, information extraction
Score: 0.0655


/usr/local/lib/python3.11/dist-packages/transformers/pipelines/question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


#### 4. Print results.

In [34]:
test_samples = [
  {
    "context": (
        "The Transformers library provides thousands of pretrained models "
        "to perform a wide array of tasks on natural language texts. "
        "These include sentiment analysis, text classification, named entity recognition, "
        "information extraction, question answering, summarization, translation, "
        "and even dialogue generation in conversational AI systems."
    ),
    "question": "List at least three types of tasks the Transformers library models can handle."
  },
  {
    "context": (
        "The Stanford Question Answering Dataset (SQuAD) is a reading comprehension dataset consisting of questions "
        "posed on a set of Wikipedia articles. The answer to each question is a segment of text from the corresponding "
        "reading passage. SQuAD version 1.1 contains over 100,000 question-answer pairs, while version 2.0 introduces "
        "unanswerable questions to better test robustness."
    ),
    "question": "What makes SQuAD 2.0 different from version 1.1?"
  },
  {
    "context": (
        "Hugging Face’s Trainer API wraps the typical training loop with features like automatic evaluation, model saving, "
        "logging, and mixed-precision training. By specifying TrainingArguments, you can control batch size, number of epochs, "
        "learning rate, checkpointing strategy, and more without writing boilerplate PyTorch loops."
    ),
    "question": "Name two capabilities provided by the Trainer API when fine-tuning models."
  }
]

for sample in test_samples:
  result = squad_qa_pipeline(sample)
  print("\nContext:\n", sample["context"])
  print("Question: ", sample["question"])
  print(f"Answer: {result['answer']} (score: {result['score']:.4f})")
  print("-" * 80)


Context:
 The Transformers library provides thousands of pretrained models to perform a wide array of tasks on natural language texts. These include sentiment analysis, text classification, named entity recognition, information extraction, question answering, summarization, translation, and even dialogue generation in conversational AI systems.
Question:  List at least three types of tasks the Transformers library models can handle.
Answer: thousands (score: 0.0963)
--------------------------------------------------------------------------------

Context:
 The Stanford Question Answering Dataset (SQuAD) is a reading comprehension dataset consisting of questions posed on a set of Wikipedia articles. The answer to each question is a segment of text from the corresponding reading passage. SQuAD version 1.1 contains over 100,000 question-answer pairs, while version 2.0 introduces unanswerable questions to better test robustness.
Question:  What makes SQuAD 2.0 different from version 1.1?


### Task III. Create a text generation pipeline with a custom model.

Write a short Python script to demonstrate the basic usage of a pipeline for text generation. Use the pipeline function to create a text generation pipeline with a custom model.

#### 1. Fine-tune a text generation model such as GPT-2 on a custom dataset (e.g., a collection of stories, articles, or other text).

##### i. Import Necessary Library

In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from transformers import pipeline
import torch

##### ii. Generate/Define Dataset

In [36]:
# sample_text = """
#   Once upon a time, in a distant kingdom, there lived a brave knight named Sir Alaric. He was known for his courage and loyalty to the crown.
#   In a faraway land, a wise old wizard discovered a magical artifact that could grant wishes, but only to those pure of heart.
#   Deep in the forest, a curious explorer stumbled upon an ancient ruin filled with treasures and forgotten secrets.
#   Long ago, a young princess set out on a quest to save her kingdom from a terrible dragon that terrorized the land.
#   In the heart of the mountains, a hidden village thrived, protected by an ancient spell cast by a powerful sorcerer.
# """
# with open("custom_stories.txt", "w") as f:
#     f.write(sample_text)

##### iii. Prepare the tokenizer and GPT2 pretrained model from HuggingFace

<a>https://huggingface.co/openai-community/gpt2</a>

In [12]:
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")

In [13]:
gpt2_model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

##### iv. Tokenize the dataset by gpt2 tokenizer

In [14]:
block_size = 64
try:
    train_dataset = TextDataset(
        tokenizer=gpt2_tokenizer,
        file_path="custom_stories.txt",
        block_size=block_size
    )
    print(f"Dataset created with {len(train_dataset)} samples.")
except Exception as e:
    print(f"Error creating dataset: {e}")
    exit()

if len(train_dataset) == 0:
    print("Error: Dataset is empty. Ensure the text file has sufficient content.")
    exit()

data_collator = DataCollatorForLanguageModeling(
    tokenizer=gpt2_tokenizer,
    mlm=False
)

Dataset created with 17 samples.


/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [15]:
train_dataset[0]

tensor([   32,  1285,  2084,   257,  1545,  9392,   257,  3155,   286,   584,
        11886,   625,   329,  8073,    13, 16178,    11,   262,  2057,   357,
         4360,   407,   262,  8237,     8,   373, 12539,   572,   262,  3084,
          329,   644,  2900,   503,   284,   307,   617, 14800,  1446,    81,
        47883,    13,   679,  8228,   262,  4811,   286,  1016,   329,   262,
        12238,    11,   517,  8119,  1573,   625,   262,  2392, 11721,  1573,
           11,   674,  2457,   711])

##### v. Set up the training arguments

In [16]:
training_gpt2_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=100,
    per_device_train_batch_size=32,
    save_steps=500,
    save_total_limit=2,
    logging_steps=10,
    dataloader_num_workers=8,
)

##### vi. Create Trainer and fine tuning the GPT2 model

In [17]:
gpt2_trainer = Trainer(
    model=gpt2_model,
    args=training_gpt2_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)
try:
    gpt2_trainer.train()
except Exception as e:
    print(f"Error during training: {e}")
    exit()

Step,Training Loss
10,3.745700
20,2.399700
30,1.410400
40,0.728500
50,0.366700
60,0.190600
70,0.120500
80,0.099500
90,0.080900
100,0.067000


##### vii. Save the fine tuned model and the tokenizer

In [18]:
gpt2_model.save_pretrained("./gpt2-finetuned")
gpt2_tokenizer.save_pretrained("./gpt2-finetuned")

('./gpt2-finetuned/tokenizer_config.json',
 './gpt2-finetuned/special_tokens_map.json',
 './gpt2-finetuned/vocab.json',
 './gpt2-finetuned/merges.txt',
 './gpt2-finetuned/added_tokens.json')

#### 2. Use the fine-tuned model with the pipeline function to generate text based on a given prompt

##### viii. Reload the fine tuned model into a pipeline

In [19]:
text_gpt2_generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer="./gpt2-finetuned",
    device= "cuda" if torch.cuda.is_available() else "cpu"
)

Device set to use cuda


In [20]:
prompt = "In a mystical land, a young hero"
generated_text = text_gpt2_generator(
    prompt,
    max_length=1,
    num_return_sequences=1,
    truncation=True,
    pad_token_id=gpt2_tokenizer.eos_token_id,
    max_new_tokens=100
)

Both `max_new_tokens` (=100) and `max_length`(=1) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


##### ix. Print text-generation results.

In [21]:
print("\nGenerated Text:")
for i, text in enumerate(generated_text):
    print(f"Sequence {i+1}: {text['generated_text']}")


Generated Text:
Sequence 1: In a mystical land, a young hero one of the strangest stories I have ever heard, it is reported that a couple of years ago a friend invited a couple of other couples over for dinner. Eventually, the food (but not the wine) was cleared off the table for what turned out to be some fierce Scrabbling. Heeding the strategy of going for the shorter, more valuable word over the longer cheaper word, our final play turned out to be a costly Scrabbling. Our final play also resulted in some fierce Sc
